# 02 - Run a VLM from Python (`pyneat.genai.VisionLanguageModel`)

A VLM answers questions about an **image**. This notebook loads a VLM, asks about an image directly,
then shows the efficient pattern - **encode the image once, ask many questions** - so you do not
re-encode the same frame for every follow-up.

> Loading a VLM takes a few minutes and several GB of memory (language weights plus a vision
> encoder). Run these cells on the DevKit.


## Which handle: `GenAIModel` vs `VisionLanguageModel`

Two handles can run a VLM:

| | `GenAIModel` | `VisionLanguageModel` |
| --- | --- | --- |
| Role | auto-detecting llm/vlm/asr | text + vision handle (used here) |
| Capability queries | `task()`, `accepts_text/image/audio()` | `accepts_image()` only |
| `run()` / `stream()` | yes | yes |
| `encode()` / `cached_image_count()` | **no** | **yes** |

Both take a **text prompt plus optional images**. `VisionLanguageModel` is "language-in always,
image-in when `accepts_image()` is `True`" — which is why it exposes `accepts_image()` but **not**
`accepts_text()` (a language model always accepts text, so there is nothing to ask). `GenAIModel`
keeps `accepts_text()` because it also fronts ASR models, which take audio rather than a text prompt.

The one reason to prefer `VisionLanguageModel` is the **encode-once / cache-image** path below
(`encode()` + `use_cached_images`). If you do not need that, `GenAIModel` works just as well for a
VLM. For the direct-vs-server choice: use a direct handle for in-process logic; use `GenAIServer`
only when the boundary is HTTP (see `../05-genai-server/`).

## Image format - a correctness detail

VLM image inputs must be **`uint8` HWC RGB** tensors. OpenCV reads images as **BGR**, so convert
before handing the array to the request. The Neat/OpenCV convention: a 3-channel `cv::Mat` is treated
as BGR and converted to RGB internally, but when you pass a **NumPy** array you convert it yourself
with `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`. Getting this wrong swaps the red/blue channels and
quietly degrades answers.


## Memory implications

- A VLM holds **two** things resident: the language model weights **and** a vision encoder.
- Encoding an image produces image embeddings. With `encode(...)` those embeddings are **cached in
  the model** so repeated questions about the same image skip re-encoding - cheaper and faster.
- `Qwen3-VL-4B-Instruct-GPTQ-a16w4` is the happy-path VLM. If disk is tight (check `df -h /` first),
  `LFM2-VL-450M-a16w4` is the smallest viable VLM.

## Step 1 - Load VLM and image

In [ ]:
import cv2
import numpy as np
import pyneat as neat

MODEL_DIR = "/media/nvme/llima/models/gemma-4-E2B-it-GPTQ-a16w4"
IMAGE_PATH = "../../tutorial/assets/images/image.png"  # any local image (relative to this notebook)

def load_rgb(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"failed to read image: {path}")
    return np.asarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))   # HWC RGB uint8

model = neat.genai.VisionLanguageModel(MODEL_DIR)
# model = neat.genai.GenAIModel(MODEL_DIR)

image = load_rgb(IMAGE_PATH)
print("model_id     :", model.model_id())
print("accepts_image:", model.accepts_image())
print("image shape  :", image.shape, image.dtype)   # (H, W, 3) uint8

**Interpretation.** `accepts_image()` must be `True` for a VLM directory. The image is a
`(H, W, 3)` `uint8` array in RGB order - exactly what the request expects.

## Step 2 - Ask with a direct image

The simplest path: attach the image to `request.images` and run once. Best for a one-shot visual
question.

In [ ]:
direct = neat.genai.GenerationRequest()
direct.prompt = "Describe this image in one sentence."
direct.images = [image]          # list of uint8 HWC RGB arrays
direct.max_new_tokens = 96

print("direct image:", model.run(direct).text)

**Interpretation.** `request.images` accepts a list, so you can pass more than one image. Under
the hood the array is converted to a Neat image tensor (BGR->RGB handled for cv::Mat; we already made
it RGB). One `run()` = one encode + one generate.

## Step 3 - Encode once, ask many times (when supported)

If you will ask several questions about the **same** image, you can `encode()` it once — this runs
the vision encoder and caches the embedding — then set `use_cached_images = True` on each follow-up
so it skips re-encoding.

**This only works on single-output vision encoders.** On a *multi-output "DeepStack"* VLM
(Qwen3-VL is the notable one), `encode()` **raises `RuntimeError`** — the encoder emits several
tensors per image, so there is nothing single to cache. It does **not** return `False`; it throws.
So guard the call and fall back to sending the image on every request (Step 2). Models like
`Qwen2.5-VL`, `LFM2-VL`, `paligemma`, and `gemma`-family VLMs use single-output encoders and support
the cache; `Qwen3-VL` does not. The only sure test is to try `encode()` and catch the exception.

In [ ]:
# encode() runs the vision encoder once and caches the embedding. It works on single-output
# vision encoders; on a multi-output "DeepStack" VLM (e.g. Qwen3-VL) it raises RuntimeError.
# Guard it and fall back to passing the image on every request.
try:
    model.encode([image])
    print("cached_images =", model.cached_image_count())
    use_cache = True
except RuntimeError as err:
    print("encode() not supported for this model:", err)
    print("-> falling back to sending the image on every request")
    use_cache = False

def ask(prompt, max_new_tokens=96):
    req = neat.genai.GenerationRequest()
    req.prompt = prompt
    req.max_new_tokens = max_new_tokens
    if use_cache:
        req.use_cached_images = True   # reuse the cached embedding
    else:
        req.images = [image]           # re-supply the image each call
    return model.run(req).text

print("q1:", ask("What details should I inspect more closely?"))
print("q2:", ask("Summarize the image in three keywords.", 48))

**Interpretation.** On success `encode()` returns `True` and `cached_image_count()` becomes `1`;
both follow-up questions then skip re-encoding. `encode()` never returns `False` — on an unsupported
(DeepStack / multi-output) model it **raises `RuntimeError`**, which is why the call is wrapped in
`try/except` rather than checked with `if not ...`. When it is unsupported, the fallback path passes
`images=[image]` on every request and pays the (cheap for one image) encode each time. Calling
`encode()` again **replaces** the cached image rather than adding to it.

## Step 4 - Image on a chat message

In a multi-turn conversation, attach the image to the specific `ChatMessage` that needs it, via
`message.images`. This keeps the image next to the exact text it belongs to.

In [ ]:
image_message = neat.genai.ChatMessage()
image_message.role = "user"
image_message.content = "What is the main subject of this image?"
image_message.images = [image]

req = neat.genai.GenerationRequest()
req.messages = [image_message]
req.max_new_tokens = 96
print("message image:", model.run(req).text)

**When to use which image path**

| Situation | Use |
| --- | --- |
| One question about one image | `request.images` (Step 2) |
| Many questions about the **same** image | `encode()` + `use_cached_images=True` (Step 3) |
| A conversation where one turn carries the image | `ChatMessage.images` (Step 4) |
| Each request uses a **different** image | direct `request.images` each time (caching gains nothing) |

## References

- [`020_run_a_vlm/run_a_vlm.py`](https://github.com/sima-neat/core/blob/main/tutorials/020_run_a_vlm/run_a_vlm.py) and its [`README.md`](https://github.com/sima-neat/core/blob/main/tutorials/020_run_a_vlm/README.md)
- [`VisionLanguageModel.h`](https://github.com/sima-neat/core/blob/main/include/genai/VisionLanguageModel.h), [`GenAITypes.h`](https://github.com/sima-neat/core/blob/main/include/genai/GenAITypes.h) — uint8 HWC RGB requirement
- [`module.cpp`](https://github.com/sima-neat/core/blob/main/python/src/module.cpp) — `encode`, `cached_image_count`, `images` bindings
- [`genai-model.mdx`](https://github.com/sima-neat/core/blob/main/docs/develop-apps/development-workflow/genai-model.mdx)